In [ ]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

# =========================
# Dynamic World + Sentinel-2
# =========================
START = "2024-01-01"
END = "2024-12-31"

dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(panama_geom)
    .filterDate(START, END)
)

dw_mode = dw_col.select("label").mode()

VIS_PALETTE = [
    "419bdf", "397d49", "88b053", "7a87c6",
    "e49635", "dfc35a", "c4281b", "a59b8f", "b39fe1"
]

Map.addLayer(
    dw_mode.clip(panama_geom),
    {"min": 0, "max": 8, "palette": VIS_PALETTE},
    "Dynamic World"
)

s2_col = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterBounds(panama_geom)
    .filterDate(START, END)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

s2_median = s2_col.median().clip(panama_geom)

Map.addLayer(
    s2_median,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    "Sentinel-2 RGB"
)

# =========================
# DEM (MERIT)
# =========================
dem = ee.Image("MERIT/DEM/v1_0_3").clip(panama_geom)

Map.addLayer(
    dem,
    {
        "min": 0,
        "max": 3500,
        "palette": [
            "#000000", "#478fcd", "#86c58e", "#afc35e",
            "#8f7131", "#b78d4f", "#e2b8a6", "#ffffff"
        ],
    },
    "Elevation"
)

# =========================
# Slope
# =========================
slope = ee.Image("projects/deforestation-495419/assets/panama_slopee").clip(panama_geom)

Map.addLayer(slope, {"min": 0, "max": 30}, "Slope")

# =========================
# Soils
# =========================
pa = ee.Image("projects/deforestation-495419/assets/Soil_Org_C").clip(panama_geom)
pn = ee.Image("projects/deforestation-495419/assets/Soil_N").clip(panama_geom)
ps = ee.Image("projects/deforestation-495419/assets/sand").clip(panama_geom)
ph = ee.Image("projects/deforestation-495419/assets/pH").clip(panama_geom)

Map.addLayer(pa, {"min": 0, "max": 150}, "Soil Organic Carbon")
Map.addLayer(pn, {"min": 0, "max": 80}, "Nitrogen")
Map.addLayer(ps, {"min": 0, "max": 1000}, "Sand")
Map.addLayer(ph, {"min": 40, "max": 80}, "pH")

# =========================
# Distqnce to Roads
# =========================
roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

roads_raster = ee.Image().byte().paint(roads, 1)

distance_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_roads_m")
    .clip(panama_geom)
)

Map.addLayer(roads_raster, {"palette": ["black"]}, "Roads")
Map.addLayer(distance_roads, {"min": 0, "max": 5000}, "Distance to Roads")

# =========================
# Forest type (dry vs wet)
# =========================

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

landcover = worldcover.select("Map")

forest = landcover.eq(10)

ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")

dry_ecoregion = ecoregions.filter(
    ee.Filter.eq("ECO_NAME", "Panamanian dry forests")
)

dry_mask = ee.Image.constant(0).paint(dry_ecoregion, 1).selfMask()

dry_forest = forest.updateMask(dry_mask)
wet_forest = forest.updateMask(dry_mask.unmask(0).Not())

classified = ee.Image(0)
classified = classified.where(wet_forest, 1)
classified = classified.where(dry_forest, 2)

Map.addLayer(
    classified.clip(panama_geom),
    {"min": 0, "max": 2, "palette": ["white", "006400", "d4a017"]},
    "Forest Type"
)

# =========================
# Distance to Deforested Area
# =========================
hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").clip(panama_geom)

lossyear = hansen.select("lossyear")

recent_loss = lossyear.gte(20).selfMask()

distance_loss = (
    recent_loss.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_loss")
    .clip(panama_geom)
)

Map.addLayer(recent_loss, {"palette": ["red"]}, "Recent Loss")
Map.addLayer(distance_loss, {"min": 0, "max": 5000}, "Distance to Loss")

# =========================
# Distance to Rural / Urban Areas
# =========================
smod = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030").select("smod_code")

urban = smod.gte(21).And(smod.lte(30))
rural = smod.gte(11).And(smod.lte(13))

urban_img = urban.unmask(0).toByte()
rural_img = rural.unmask(0).toByte()

distance_urban = urban_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)
distance_rural = rural_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)

Map.addLayer(distance_urban, {"min": 0, "max": 25000}, "Distance Urban")
Map.addLayer(distance_rural, {"min": 0, "max": 25000}, "Distance Rural")

Map.addLayer(urban.selfMask(), {"palette": ["red"]}, "Urban")
Map.addLayer(rural.selfMask(), {"palette": ["blue"]}, "Rural")

# =========================
# Monthly Precipitation DAta
# =========================
chirps = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_SAT")
    .filterDate("2018-05-01", "2018-05-31")
    .select("precipitation")
)

precip = chirps.mean().clip(panama_geom)

Map.addLayer(
    precip,
    {
        "min": 1,
        "max": 17,
        "palette": ["#001137", "#0aab1e", "#e7eb05", "#2c7fb8", "#253494"]
    },
    "Precipitation"
)

# =========================
# Monthyly Temperature Data
# =========================
temp = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2023-01-01", "2023-02-01")
    .first()
    .clip(panama_geom)
)

Map.addLayer(
    temp,
    {
        "bands": ["temperature_2m"],
        "min": 250,
        "max": 320
    },
    "Temperature"
)

# =========================
# Distance to Hydrological Basins
# =========================
basins = ee.Image("projects/deforestation-495419/assets/hydrographic_basins").clip(panama_geom)

basins_int = basins.toInt()
neighbors = basins_int.focal_mode(1)

edges = basins_int.neq(neighbors).selfMask()

dist_basin = (
    edges.fastDistanceTransform(512).sqrt()
    .multiply(basins.projection().nominalScale())
    .clip(panama_geom)
)

Map.addLayer(basins, {}, "Basins")
Map.addLayer(edges, {"palette": ["red"]}, "Basin Edges")
Map.addLayer(dist_basin, {"min": 0, "max": 50000}, "Distance Basin")

# =========================
# Distance to Protected Areas
# =========================
pa_fc = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(panama_geom)

pa_raster = ee.Image().byte().paint(pa_fc, 1)

dist_pa = (
    pa_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .clip(panama_geom)
)

Map.addLayer(pa_fc, {}, "Protected Areas")
Map.addLayer(dist_pa, {}, "Distance Protected Areas")

# Final Map Control
Map.addLayerControl()

Map

Map(center=[8.515838945899919, -80.10966640141515], controls=(WidgetControl(options=['position', 'transparent_…